In [40]:
import pandas as pd
import db_mgmt as mgmt
import os
import sqlite3

In [41]:
def replace_table(db, data):
    conn = sqlite3.connect(db)
    for table, df in data.items():
        if not df.empty:
            conn.execute(f"DELETE FROM {table}")
            df.to_sql(table, conn, if_exists='append', index=False)
        
    conn.commit()
    conn.close()

In [42]:
data_schema = 'dbs/canoe_dataset_schema 5.sql'
db_file = 'dbs/canoe_transport.sqlite'
os.remove(db_file) if os.path.exists(db_file) else None
mgmt.convert_sql_to_sqlite(data_schema, db_file)
data = mgmt.sqlite_to_dfs(db_file)

dir = 'transport/inputs/'
transp = {}
for file in os.listdir(dir):
    #print(file)
    if file.endswith('.csv'):
        print(file)
        transp[file.split('_')[0]] = pd.read_csv(dir + file)

tech_to_remove = pd.read_csv('transport/Fuel_techs.csv')['tech'].to_list()
new_transp = {}
fuel = {}
transport = {}

for table, df in transp.items():
    if 'tech' in df.columns:
        fuel[table] = df[df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
        new_transp[table] = df[~df['tech'].isin(tech_to_remove)].copy().reset_index(drop=True)
    else:
        new_transp[table] = df.copy()

comm = pd.read_csv('transport/Fuel_comm.csv')['commodity'].to_list()

DataSource_transport.csv
ExistingCapacity_transport.csv
DataSet_transport.csv
CapacityFactorTech_transport.csv
LimitTechInputSplitAnnual_transport.csv
LimitAnnualCapacityFactor_transport.csv
CostFixed_transport.csv
TechGroupMember_transport.csv
Technology_transport.csv
EmissionActivity_transport.csv
LifetimeTech_transport.csv
CostInvest_transport.csv
Demand_transport.csv
CapacityToActivity_transport.csv
LifetimeSurvivalCurve_vehicle.csv
Commodity_transport.csv
CostVariable_transport.csv
TechGroup_transport.csv
Efficiency_transport.csv


In [43]:
new_transp['Commodity'] = pd.read_csv('transport/Commodity.csv')
new_transp['CostFixed'] = pd.concat([pd.read_csv('dbs/CostFixed.csv'), new_transp['CostFixed']], ignore_index=True)
new_transp['CostInvest'] = pd.concat([pd.read_csv('dbs/CostInvest.csv'), new_transp['CostInvest']], ignore_index=True)
fuel['Commodity'] = pd.read_csv('transport/Commodity_fuel.csv')

In [44]:
for table, df in new_transp.items():
    for c in comm:
        if c in df.values:
            new_transp[table] = df[~df.isin([c]).any(axis=1)].copy().reset_index(drop=True)

for table, df in fuel.items():
    for c in comm:
        if c in df.values:
            fuel[table] = df[df.isin([c]).any(axis=1)].copy().reset_index(drop=True)

for table, df in new_transp.items():
    print(table)
    if ('data_id' in data[table].columns) and (table != 'DataSet'):
        if 'region' not in df.columns:
            df['data_id'] = 'TRPHR002'
        else:
            df['data_id'] = 'TRPHR' + df['region'].astype(str) + '002'

DataSource
ExistingCapacity
DataSet
CapacityFactorTech
LimitTechInputSplitAnnual
LimitAnnualCapacityFactor
CostFixed
TechGroupMember
Technology
EmissionActivity
LifetimeTech
CostInvest
Demand
CapacityToActivity
LifetimeSurvivalCurve
Commodity
CostVariable
TechGroup
Efficiency


In [45]:
df = new_transp['LifetimeTech'].copy()

new = df.loc[df['tech'].str.endswith('_N'), 'tech'].to_list()
replacer = {tech: tech[:-2] for tech in new}

ex = df.loc[df['tech'].str.endswith('_EX'), 'tech'].to_list()
replacer.update({tech: tech[:-3] for tech in ex})
replacer

df = df.replace(replacer)

df.drop_duplicates(subset=['region', 'tech'])

new_transp['LifetimeTech'] = df.copy()
mgmt.update_sqlite(db_file, new_transp)


Inserting into DataSource with columns: source_id,source,notes,data_id
Inserting into ExistingCapacity with columns: region,tech,vintage,capacity,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into DataSet with columns: data_id,label,version,description,status,author,date,parent_id,changelog,notes
Inserting into CapacityFactorTech with columns: region,period,season,tod,tech,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitTechInputSplitAnnual with columns: region,period,input_comm,tech,operator,proportion,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into LimitAnnualCapacityFactor with columns: region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into CostFixed with columns: region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
Inserting into TechGroupMem

In [46]:
df = new_transp['LimitAnnualCapacityFactor'].copy()

df[df['factor']>1]

,region,tech,vintage,output_comm,operator,factor,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id


In [47]:
# db_file = 'dbs/canoe_hr_16d.sqlite'


data = mgmt.sqlite_to_dfs(db_file)
df_c = data['CostFixed']
df_e = data['Efficiency']


# Create a mask of rows in df_c that are NOT in df_e
# We use .merge or .isin on a combined string/tuple for multiple columns
cols = ['region', 'tech', 'vintage']

# This identifies which rows in df_c are NOT found in df_e
missing_in_e = df_c[~df_c[cols].apply(tuple, axis=1).isin(df_e[cols].apply(tuple, axis=1))]

# print(missing_in_e[cols])


rows_list = []
new_vintage = []

for r in missing_in_e['region'].unique():
    for t in missing_in_e['tech'].unique():
        sub_df = df_e.loc[(df_e['region'] == r) & (df_e['tech'] == t)]
        for v in missing_in_e.loc[(missing_in_e['region']==r) & (missing_in_e['tech']==t), 'vintage'].unique():
            if not sub_df.empty:
                # .iloc[[0]] returns a DataFrame with 1 row, preserving columns correctly
                rows_list.append(sub_df.iloc[[0]])
                new_vintage.append(v)
            

new_transp['CostFixed']

# Concat everything at once - much faster!
to_add = pd.concat(rows_list, ignore_index=True)
to_add['vintage'] = new_vintage

mgmt.update_sqlite(db_file, {'Efficiency': to_add})

Inserting into Efficiency with columns: region,input_comm,tech,vintage,output_comm,efficiency,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id


In [48]:

data = mgmt.sqlite_to_dfs(db_file)
df_ex = data['ExistingCapacity']
df_ef = data['Efficiency']


In [ ]:
lookup = df_ex.set_index(['region', 'tech', 'vintage']).index.to_list()


df_ef_sub = df_ef.loc[df_ef['vintage'] < 2025].copy()


index_to_rm = []
for idx, row in df_ef_sub.iterrows():
    key = (row['region'], row['tech'], row['vintage'])
    if key not in lookup:
        #print(key)
        index_to_rm.append(idx)

new_df_ef = df_ef.drop(index_to_rm)

replace_table(db_file, {'Efficiency': new_df_ef})

('ON', 'T_HDV_CHRG', 2020)
('ON', 'T_LDV_BEV_CHRG', 2020)
('ON', 'T_LDV_PHEV_CHRG', 2020)
('AB', 'T_HDV_CHRG', 2020)
('AB', 'T_LDV_BEV_CHRG', 2020)
('AB', 'T_LDV_PHEV_CHRG', 2020)
('BC', 'T_HDV_CHRG', 2020)
('BC', 'T_LDV_BEV_CHRG', 2020)
('BC', 'T_LDV_PHEV_CHRG', 2020)
('QC', 'T_HDV_CHRG', 2020)
('QC', 'T_LDV_BEV_CHRG', 2020)
('QC', 'T_LDV_PHEV_CHRG', 2020)
('MB', 'T_HDV_CHRG', 2020)
('MB', 'T_LDV_BEV_CHRG', 2020)
('SK', 'T_HDV_CHRG', 2020)
('SK', 'T_LDV_BEV_CHRG', 2015)
('SK', 'T_LDV_BEV_CHRG', 2020)
('NLLAB', 'T_HDV_CHRG', 2020)
('NLLAB', 'T_LDV_BEV_CHRG', 2020)
('NLLAB', 'T_LDV_PHEV_CHRG', 2020)
('PEI', 'T_HDV_CHRG', 2020)
('PEI', 'T_LDV_BEV_CHRG', 2020)
('PEI', 'T_LDV_PHEV_CHRG', 2020)
('NS', 'T_HDV_CHRG', 2020)
('NS', 'T_LDV_BEV_CHRG', 2020)
('NS', 'T_LDV_PHEV_CHRG', 2020)
('NB', 'T_HDV_CHRG', 2020)
('NB', 'T_LDV_BEV_CHRG', 2020)
('NB', 'T_LDV_PHEV_CHRG', 2020)


In [51]:
data = mgmt.sqlite_to_dfs(db_file)

df_sc = data['LifetimeSurvivalCurve']
df_lt = data['LifetimeTech']

df_sc['period_diff'] = df_sc['period'] - df_sc['vintage']

filtered_df = df_sc[df_sc['fraction'] == 0].copy()

new_SC = filtered_df[
    filtered_df.groupby(['region', 'tech'])['period_diff'].transform('min') == filtered_df['period_diff']
]

NEW_SC = pd.concat([df_sc.loc[df_sc['fraction'] != 0],new_SC], ignore_index=True)
NEW_SC = NEW_SC.drop(columns=['period_diff'])

replace_table(db_file, {'LifetimeSurvivalCurve': NEW_SC})


lookup = new_SC.drop_duplicates(subset=['region', 'tech']).set_index(['region', 'tech'])['period_diff']

mask = df_lt.set_index(['region', 'tech']).index.isin(lookup.index)

df_lt.loc[mask, 'lifetime'] = df_lt[mask].set_index(['region', 'tech']).index.map(lookup)

mgmt.update_sqlite(db_file, { 'LifetimeTech': df_lt})


Inserting into LifetimeTech with columns: region,tech,lifetime,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id


In [52]:
data = mgmt.sqlite_to_dfs(db_file)
data['CostFixed']

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,ON,2025,T_HDV_CHRG,2020,1.643260,None,None,None,None,None,None,None,None,TRPHRON002
1,ON,2030,T_HDV_CHRG,2020,1.643260,None,None,None,None,None,None,None,None,TRPHRON002
2,ON,2050,T_HDV_CHRG,2040,1.643260,None,None,None,None,None,None,None,None,TRPHRON002
3,ON,2050,T_HDV_CHRG,2045,1.643260,None,None,None,None,None,None,None,None,TRPHRON002
4,ON,2050,T_HDV_CHRG,2050,1.643260,None,None,None,None,None,None,None,None,TRPHRON002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
949,SK,2040,T_MDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
950,SK,2040,T_MDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
951,SK,2045,T_MDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
952,SK,2045,T_MDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002


In [53]:

data = mgmt.sqlite_to_dfs(db_file)

ex = data['ExistingCapacity']
cf = data['CostFixed']


lookup = ex.drop_duplicates(subset=['region', 'tech', 'vintage']).set_index(['region', 'tech', 'vintage'])
# print(lookup)

old_cf = cf.loc[cf['vintage'] < 2025].copy()

future_cf = cf.loc[cf['vintage'] >= 2025].copy()

NEW_CF = pd.DataFrame()
for i, row in old_cf.iterrows():
    key = (row['region'], row['tech'], row['vintage'])
    if key in lookup.index:
        NEW_CF = pd.concat([NEW_CF, row.to_frame().T], ignore_index=True)

NEW_CF = pd.concat([NEW_CF, future_cf], ignore_index=True)

replace_table(db_file, {'CostFixed': NEW_CF})

In [54]:
data = mgmt.sqlite_to_dfs(db_file)
data['CostFixed']

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRAB002
1,AB,2025,T_LDV_BEV_CHRG,2015,0.267432,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRAB002
2,AB,2025,T_LDV_PHEV_CHRG,2015,0.267432,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRAB002
3,BC,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRBC002
4,BC,2025,T_LDV_BEV_CHRG,2015,0.267432,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRBC002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892,SK,2040,T_MDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
893,SK,2040,T_MDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
894,SK,2045,T_MDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
895,SK,2045,T_MDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002


In [55]:
cv = data['CostVariable']

lookup = ex.drop_duplicates(subset=['region', 'tech', 'vintage']).set_index(['region', 'tech', 'vintage'])

# 1. Identify rows NOT in lookup (you already did this)
mask_not_in_lookup = ~cv.set_index(['region', 'tech', 'vintage']).index.isin(lookup.index)

# 2. Identify rows where vintage is less than 2025
mask_old_vintage = cv['vintage'] < 2025

# 3. Combine them: Rows that are BOTH missing from lookup AND have vintage < 2025
# These are the rows you want to DISCARD
rows_to_drop_mask = mask_not_in_lookup & mask_old_vintage

# 4. Create NEW_CF by keeping everything ELSE
NEW_CV = cv[~rows_to_drop_mask].copy()


replace_table(db_file, {'CostVariable': NEW_CV})


In [56]:
data = mgmt.sqlite_to_dfs(db_file)


for table, df in data.items():
    if 'vintage' in df.columns:
        data[table] = df.loc[df['vintage']< 2050]

for table, df, in data.items():
    if (table not in ['TimePeriod', 'LifetimeSurvivalCurve'])  and 'period' in df.columns:
        data[table] = data[table].loc[data[table]['period'] < 2050]


replace_table(db_file, data)

In [57]:
data = mgmt.sqlite_to_dfs(db_file)

cap_f = data['LimitAnnualCapacityFactor'].copy()
lft = data['LifetimeTech'].copy()

df = data['TimePeriod'].copy()
start = 2025 # Hardcoded for now, but could be dynamic based on TimePeriod table


lookup = lft.drop_duplicates(['region', 'tech']).set_index(['region', 'tech']).index
to_be_rm = []
for i in lookup:
    # Filter the data first
    vint_series = cap_f.loc[(cap_f['region'] == i[0]) & (cap_f['tech'] == i[1]), 'vintage']
    lt_series = lft.loc[(lft['region'] == i[0]) & (lft['tech'] == i[1]), 'lifetime']
    ex_ = ex.loc[(ex['region']== i[0]) & (ex['tech']==i[1])]

    # Only proceed if BOTH dataframes actually have the data
    if not vint_series.empty and not lt_series.empty:
        vint = vint_series.iat[0]
        lt_ = lt_series.iat[0]
        if vint < (start - lt_):
            to_be_rm.extend(cap_f.loc[(cap_f['region'] == i[0]) & (cap_f['tech'] == i[1]), 'vintage'].index.values)
            # print(f"Removing {i} with vintage {vint} and lifetime {lt_}")


data['LimitAnnualCapacityFactor'] = cap_f.loc[~cap_f.index.isin(to_be_rm)]

# print(data['LimitAnnualCapacityFactor'])


replace_table(db_file, data)



In [58]:
df_sc = data['LifetimeSurvivalCurve'].copy()
df_lf = data['LifetimeTech'].copy()

vars = df_sc.drop_duplicates(subset=['region', 'tech', 'vintage'])[['region', 'tech', 'vintage']].reset_index(drop=True)

for idx, row in vars.iterrows():
    region = row['region']
    tech = row['tech']
    vintage = row['vintage']
    df = df_sc.loc[(df_sc['tech']==tech)&(df_sc['region']==region)&(df_sc['vintage']==vintage)].copy()
    new_lt = (df['period']- df['vintage']).max()

    df_lf.loc[(df_lf['tech']==tech)&(df_lf['region']==region), 'lifetime'] = new_lt


In [59]:
data['LifetimeSurvivalCurve'] = df_sc
data['LifetimeTech'] = df_lf

replace_table(db_file, data)


In [60]:
df = data['CostFixed'].copy()
df.loc[df['tech'].str.contains('HDV_CH')]

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRAB002
3,BC,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRBC002
6,MB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRMB002
8,NB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRNB002
11,NLLAB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRNLLAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,SK,2040,T_HDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
737,SK,2040,T_HDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
738,SK,2045,T_HDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
739,SK,2045,T_HDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002


In [61]:

df_cv = data['CostVariable'].copy()
df_cf = data['CostFixed'].copy()
df_lf = data['LifetimeTech'].copy()

vars = df_cv.drop_duplicates(subset=['region', 'tech', 'vintage'])[['region', 'tech', 'vintage']].reset_index(drop=True)


index_to_rm = []
for idx, row in vars.iterrows():
    region = row['region']
    tech = row['tech']
    vintage = row['vintage']
    
    lf = df_lf.loc[(df_lf['tech']==tech)&(df_lf['region']==region), 'lifetime']
    if not lf.empty:
        lt_ = lf.iat[0]

        df = df_cv.loc[(df_cv['tech']==tech)&(df_cv['region']==region)&(df_cv['vintage']==vintage)].copy()
        df['period_diff'] = df['period']- df['vintage']
        index_to_rm.extend(df[df['period_diff'] > lt_].index.values)

data['CostVariable'] = df_cv.loc[~df_cv.index.isin(index_to_rm)].copy()

vars_cf = df_cf.drop_duplicates(subset=['region', 'tech', 'vintage'])[['region', 'tech', 'vintage']].reset_index(drop=True)


index_to_rm = []
for idx, row in vars_cf.iterrows():
    region = row['region']
    tech = row['tech']
    vintage = row['vintage']
    
    lf = df_lf.loc[(df_lf['tech']==tech)&(df_lf['region']==region), 'lifetime']
    if not lf.empty:
        lt_ = lf.iat[0]

        df = df_cf.loc[(df_cf['tech']==tech)&(df_cf['region']==region)&(df_cf['vintage']==vintage)].copy()
        df['period_diff'] = df['period']- df['vintage']
        should_rm = df[df['period_diff'] > lt_].index.values
        index_to_rm.extend(should_rm)
        if should_rm.size > 0:
            print(tech, region, vintage, lt_, index_to_rm)

data['CostFixed'] = df_cf.loc[~df_cf.index.isin(index_to_rm)].copy()


replace_table(db_file, data)

In [62]:
df = data['CostFixed'].copy()
df.loc[df['tech'].str.contains('HDV_CH')]

,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,AB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRAB002
3,BC,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRBC002
6,MB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRMB002
8,NB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRNB002
11,NLLAB,2025,T_HDV_CHRG,2015,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRNLLAB002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,SK,2040,T_HDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
737,SK,2040,T_HDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
738,SK,2045,T_HDV_CHRG,2035,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002
739,SK,2045,T_HDV_CHRG,2040,0.093601,2020 CAD (M$/PJ/year),"Obtained directly from OEO estimates, assuming...",T24,None,None,None,None,None,TRPHRSK002


In [66]:
db_file = 'dbs/canoe_hr_16d.sqlite'

data = mgmt.sqlite_to_dfs(db_file) 

for table, df in data.items():
    if 'vintage' in df.columns:
        data[table] = df.loc[df['vintage']< 2050]

for table, df, in data.items():
    if (table not in ['TimePeriod', 'LifetimeSurvivalCurve'])  and 'period' in df.columns:
        data[table] = data[table].loc[data[table]['period'] < 2050]

replace_table(db_file, data)

In [64]:
db_file = 'dbs/canoe_hr_16d.sqlite'

data = mgmt.sqlite_to_dfs(db_file) 



In [65]:
for table, df in data.items():
    if 2050 in df.values:
        print(table)

TimeSegmentFraction
LifetimeSurvivalCurve
TimePeriod
TimeSeason
TimeSeasonSequential


In [ ]:
# import sqlite3
# import pandas as pd

# # 1. Load data
# data = mgmt.sqlite_to_dfs('dbs/canoe_hr_16d.sqlite')
# region = 'ON'

# # 2. Filter data
# filtered_data = {}
# for table, df in data.items():
#     if 'region' in df.columns:
#         filtered_data[table] = df.loc[df['region'] == region].copy()
#     else:
#         filtered_data[table] = df.copy()

# # 3. Write to destination
# conn = sqlite3.connect('dbs/canoe_hr_16d.sqlite')

# for table, df in filtered_data.items():
#     print(f"Processing {table}...")
#     try:
#         # 'replace' will drop the table and recreate it. 
#         # If you need to keep specific SQL constraints/indexes, 
#         # use 'append' but handle the DELETE more carefully.
#         df.to_sql(table, conn, if_exists='replace', index=False)
        
#     except Exception as e:
#         print(f"Error processing table {table}: {e}")

# conn.close()
# print("Transfer complete.")

In [ ]:
os.getcwd()